Importing Required Header files

In [13]:
# Importing necessary libraries for data manipulation and analysis
import pandas as pd
import os

In [14]:

data_path = "../../Data/raw_datasets/"
files = sorted(os.listdir(data_path))

all_dfs = []

Data sets reading

In [15]:
# Setting the path to the directory containing the raw datasets and listing all the files in that directory. We will then read each CSV file into a pandas DataFrame and perform some basic checks on the data to ensure that it is clean and ready for analysis.
data_path = "../../data/raw_datasets/"
files = sorted(os.listdir(data_path))

for file in files:
    if file.endswith(".csv"):
    
        df = pd.read_csv(data_path + file)

        print("Shape:", df.shape)
       

Shape: (33165, 34)
Shape: (55275, 34)
Shape: (77385, 34)
Shape: (99495, 34)
Shape: (121605, 34)
Shape: (143715, 34)
Shape: (138187, 34)
Shape: (132660, 34)
Shape: (127132, 34)
Shape: (121605, 34)
Shape: (77385, 34)


Altering Order date for null values

In [16]:
# Checking for missing values in the dataset to ensure that there are no null values that could affect our analysis. This will help us identify any potential issues with the data and allow us to take appropriate steps to handle them.
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce', format='mixed')

# check first
missing_dates = df['order_date'].isna().sum()
print("Missing dates:", missing_dates)

Missing dates: 0


Data Cleaning

In [17]:
# Now we will define a function to clean the data. This function will perform various cleaning steps such as standardizing column names, handling missing values, and correcting data types. We will apply this function to our dataset to ensure that it is clean and ready for analysis.

def clean_data(df):


    df = df.copy()
   
    # STANDARDIZE COLUMN NAMES
  
    df.columns = df.columns.str.strip().str.lower()

    # DATE CLEANING

    if 'order_date' in df.columns:
        df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce', format='mixed')        
        df = df.dropna(subset=['order_date'])

   
    # PRICE CLEANING
    
    if 'original_price_inr' in df.columns:
        df['original_price_inr'] = (
            df['original_price_inr']
            .astype(str)
            .str.replace(',', '')
            .str.replace('₹', '')
        )
        df['original_price_inr'] = pd.to_numeric(df['original_price_inr'], errors='coerce')

   
    # CUSTOMER RATING
   
    if 'customer_rating' in df.columns:
        df['customer_rating'] = (
            df['customer_rating']
            .astype(str)
            .str.replace(' stars', '')
            .str.replace('/5', '')
        )
        df['customer_rating'] = pd.to_numeric(df['customer_rating'], errors='coerce')

    
    # CITY CLEANING (RULE-BASED)

    if 'customer_city' in df.columns:
        df['customer_city'] = df['customer_city'].astype(str).str.strip().str.lower()

        city_map = {
            'mumba': 'mumbai',
            'bombay': 'mumbai',
            'bangalore': 'bengaluru',
            'banglore': 'bengaluru',
            'bengalore': 'bengaluru',
            'madras': 'chennai',
            'chenai': 'chennai',
            'delhi ncr': 'delhi'
        }

        df['customer_city'] = df['customer_city'].replace(city_map)
        df['customer_city'] = df['customer_city'].str.title()

    
    # CATEGORY CLEANING
   
    if 'category' in df.columns:
        df['category'] = df['category'].astype(str).str.strip().str.lower()

        df['category'] = df['category'].replace({
            'electronic': 'electronics',
            'electronicss': 'electronics',
            'electronics & accessories': 'electronics'
        })

        df['category'] = df['category'].str.title()

 
    # BOOLEAN STANDARDIZATION
  
    bool_cols = ['is_prime_member', 'is_festival_sale', 'is_prime_eligible']

    for col in bool_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.lower()

            df[col] = df[col].map({
                'yes': True, 'y': True, '1': True, 'true': True,
                'no': False, 'n': False, '0': False, 'false': False
            })

            df[col] = df[col].fillna(False)
   
    # DELIVERY DAYS
    if 'delivery_days' in df.columns:
        df['delivery_days'] = df['delivery_days'].astype(str).str.extract('(\d+)')
        df['delivery_days'] = pd.to_numeric(df['delivery_days'], errors='coerce')

    
    # MISSING VALUES
   
    if 'delivery_charges' in df.columns:
        df['delivery_charges'] = df['delivery_charges'].fillna(0)

    if 'customer_rating' in df.columns:
        df['customer_rating'] = df['customer_rating'].fillna(df['customer_rating'].median())

    if 'customer_age_group' in df.columns:
        df['customer_age_group'] = df['customer_age_group'].fillna('Unknown')

    if 'festival_name' in df.columns:
        df['festival_name'] = df['festival_name'].fillna('No Festival')

    if 'delivery_days' in df.columns:
        df['delivery_days'] = df['delivery_days'].fillna(df['delivery_days'].median())

    # FIX ORIGINAL PRICE (LOGIC BASED)

    if 'original_price_inr' in df.columns and 'discounted_price_inr' in df.columns:
        df['original_price_inr'] = df['original_price_inr'].fillna(
            df.apply(lambda x:
                x['discounted_price_inr'] / (1 - x['discount_percent']/100)
                if x['discount_percent'] != 100 else x['discounted_price_inr'],
                axis=1
            )
        )

    return df

Pipeline building for all year's datasets

In [18]:
# Now we will apply the cleaning function to all the datasets and combine them into a single DataFrame for analysis.

for file in files:
    if file.endswith('.csv'):

        print(f"Processing {file}...")

        df=pd.read_csv(data_path + file)
        df=clean_data(df)
        all_dfs.append(df)

Processing amazon_india_2015.csv...
Processing amazon_india_2016.csv...
Processing amazon_india_2017.csv...
Processing amazon_india_2018.csv...
Processing amazon_india_2019.csv...
Processing amazon_india_2020.csv...
Processing amazon_india_2021.csv...
Processing amazon_india_2022.csv...
Processing amazon_india_2023.csv...
Processing amazon_india_2024.csv...
Processing amazon_india_2025.csv...


Shape of the cleaned dataset

In [ ]:
# After cleaning all the datasets, we will identify the common columns across all DataFrames and retain only those columns for the final combined DataFrame. This will ensure that we have a consistent set of features for analysis across all the datasets.
common_cols = set.intersection(*(set(df.columns) for df in all_dfs))
all_dfs = [df[list(common_cols)] for df in all_dfs]

final_df = pd.concat(all_dfs, ignore_index=True)
print(final_df.shape)

(1127609, 34)


In [20]:
print(len(all_dfs))

11


In [21]:
print("Total rows after combine:", final_df.shape[0])

Total rows after combine: 1127609


In [22]:
print(final_df.shape)

(1127609, 34)


In [23]:
final_df.duplicated().sum()

np.int64(0)

In [24]:
all_dfs = []

In [25]:
final_df.isnull().sum().sort_values(ascending=False)

customer_city             0
order_date                0
is_festival_sale          0
discounted_price_inr      0
subtotal_inr              0
is_prime_eligible         0
discount_percent          0
original_price_inr        0
category                  0
customer_tier             0
return_status             0
product_name              0
customer_age_group        0
product_rating            0
order_month               0
customer_spending_tier    0
product_weight_kg         0
order_year                0
festival_name             0
payment_method            0
transaction_id            0
customer_id               0
delivery_type             0
customer_rating           0
product_id                0
quantity                  0
subcategory               0
delivery_charges          0
customer_state            0
order_quarter             0
final_amount_inr          0
brand                     0
delivery_days             0
is_prime_member           0
dtype: int64

In [26]:
final_df.dtypes

customer_city                     object
customer_tier                     object
delivery_days                    float64
brand                             object
final_amount_inr                 float64
order_quarter                      int64
customer_state                    object
delivery_charges                 float64
subcategory                       object
quantity                           int64
product_id                        object
customer_rating                  float64
delivery_type                     object
customer_id                       object
transaction_id                    object
payment_method                    object
festival_name                     object
order_year                         int64
product_weight_kg                float64
is_festival_sale                    bool
discounted_price_inr             float64
subtotal_inr                     float64
is_prime_eligible                   bool
discount_percent                 float64
original_price_i

In [27]:
final_df['delivery_days'] = final_df['delivery_days'].astype('Int64')

In [ ]:
# Converting categorical columns to 'category' data type to optimize memory usage and improve performance during analysis. This will help us efficiently handle categorical data and speed up our computations.
cat_cols = [
    'category', 'subcategory', 'brand',
    'customer_city', 'customer_state',
    'payment_method', 'customer_tier',
    'customer_spending_tier'
]

for col in cat_cols:
    final_df[col] = final_df[col].astype('category')

In [29]:
final_df.shape

(1127609, 34)

In [30]:
final_df.isnull().sum().sort_values(ascending=False)

customer_city             0
order_date                0
is_festival_sale          0
discounted_price_inr      0
subtotal_inr              0
is_prime_eligible         0
discount_percent          0
original_price_inr        0
category                  0
customer_tier             0
return_status             0
product_name              0
customer_age_group        0
product_rating            0
order_month               0
customer_spending_tier    0
product_weight_kg         0
order_year                0
festival_name             0
payment_method            0
transaction_id            0
customer_id               0
delivery_type             0
customer_rating           0
product_id                0
quantity                  0
subcategory               0
delivery_charges          0
customer_state            0
order_quarter             0
final_amount_inr          0
brand                     0
delivery_days             0
is_prime_member           0
dtype: int64

In [31]:
final_df.duplicated().sum()

np.int64(0)

In [32]:
(final_df['final_amount_inr'] >= final_df['subtotal_inr']).all()

np.True_

In [33]:
(final_df[['final_amount_inr', 'original_price_inr']] < 0).sum()

final_amount_inr         0
original_price_inr    2792
dtype: int64

In [34]:
final_df['delivery_days'].describe()

count    1127609.0
mean      3.333039
std       1.748463
min            0.0
25%            2.0
50%            3.0
75%            4.0
max           15.0
Name: delivery_days, dtype: Float64

In [35]:
final_df['payment_method'].unique()
final_df['category'].unique()

['Electronics']
Categories (1, object): ['Electronics']

In [ ]:
# After cleaning and combining all the datasets, we will perform some exploratory data analysis (EDA) to understand the key trends and patterns in the data. This will help us identify any potential insights that can inform our marketing strategies and optimize our sales performance.
import pandas as pd
import os

data_path = "../../data/raw_datasets/"
files = sorted(os.listdir(data_path))

for file in files:
    if file.endswith(".csv"):
        print(f"DATASET: {file}")
        
        df = pd.read_csv(data_path + file)
        
        if 'category' in df.columns:
            print("Unique Categories:")
            print(sorted(df['category'].dropna().unique()))

DATASET: amazon_india_2015.csv
Unique Categories:
['ELECTRONICS', 'Electronic', 'Electronics', 'Electronics & Accessories', 'Electronicss']
DATASET: amazon_india_2016.csv
Unique Categories:
['ELECTRONICS', 'Electronic', 'Electronics', 'Electronics & Accessories', 'Electronicss']
DATASET: amazon_india_2017.csv
Unique Categories:
['ELECTRONICS', 'Electronic', 'Electronics', 'Electronics & Accessories', 'Electronicss']
DATASET: amazon_india_2018.csv
Unique Categories:
['ELECTRONICS', 'Electronic', 'Electronics', 'Electronics & Accessories', 'Electronicss']
DATASET: amazon_india_2019.csv
Unique Categories:
['ELECTRONICS', 'Electronic', 'Electronics', 'Electronics & Accessories', 'Electronicss']
DATASET: amazon_india_2020.csv
Unique Categories:
['ELECTRONICS', 'Electronic', 'Electronics', 'Electronics & Accessories', 'Electronicss']
DATASET: amazon_india_2021.csv
Unique Categories:
['ELECTRONICS', 'Electronic', 'Electronics', 'Electronics & Accessories', 'Electronicss']
DATASET: amazon_indi

In [ ]:
# Standardizing category names to ensure consistency across the dataset. This will help us accurately analyze the contribution of different categories to total revenue and identify key trends in customer preferences.
if 'category' in df.columns:
    df['category'] = (
        df['category']
        .astype(str)
        .str.strip()
        .str.lower()
    )

    df['category'] = df['category'].replace({
        'electronic': 'electronics',
        'electronicss': 'electronics',
        'electronics & accessories': 'electronics'
    })

    df['category'] = df['category'].str.title()

In [ ]:
# After standardizing category names, we will identify the unique categories across all datasets to understand the diversity of products being sold. This will help us identify key categories and optimize our product offerings and marketing strategies accordingly.
all_categories = set()

for file in files:
    if file.endswith(".csv"):
        df = pd.read_csv(data_path + file)
        
        if 'category' in df.columns:
            cats = df['category'].dropna().unique()
            all_categories.update(cats)

print("\nALL UNIQUE CATEGORIES ACROSS DATASETS:\n")
print(sorted(all_categories))


ALL UNIQUE CATEGORIES ACROSS DATASETS:

['ELECTRONICS', 'Electronic', 'Electronics', 'Electronics & Accessories', 'Electronicss']


In [39]:
print(sorted(final_df['payment_method'].unique()))

['BNPL', 'COD', 'Credit Card', 'Debit Card', 'Net Banking', 'UPI', 'Wallet']


In [40]:
if 'payment_method' in df.columns:
    df['payment_method'] = (
        df['payment_method']
        .astype(str)
        .str.strip()
        .str.title()
    )

In [41]:
payment_group_map = {
    'Cod': 'Cash',
    'Upi': 'Digital',
    'Credit Card': 'Digital',
    'Debit Card': 'Digital',
    'Net Banking': 'Digital',
    'Wallet': 'Digital',
    'Bnpl': 'Credit'
}

df['payment_group'] = df['payment_method'].map(payment_group_map)

In [42]:
print(sorted(final_df['customer_city'].unique())[:50])

['Ahmedabad', 'Aligarh', 'Allahabad', 'Bareilly', 'Bengaluru', 'Bhubaneswar', 'Calcutta', 'Chandigarh', 'Chennai', 'Coimbatore', 'Delhi', 'Gorakhpur', 'Hyderabad', 'Indore', 'Jaipur', 'Kanpur', 'Kochi', 'Kolkata', 'Lucknow', 'Ludhiana', 'Meerut', 'Moradabad', 'Mumbai', 'Nagpur', 'New Delhi', 'Patna', 'Pune', 'Saharanpur', 'Surat', 'Vadodara', 'Varanasi', 'Visakhapatnam']


In [43]:
if 'customer_city' in df.columns:
    df['customer_city'] = (
        df['customer_city']
        .astype(str)
        .str.strip()
        .str.title()
    )

    df['customer_city'] = df['customer_city'].replace({
        'Calcutta': 'Kolkata',
        'New Delhi': 'Delhi'
    })

In [44]:
print("Total unique:", final_df['customer_city'].nunique())
print(sorted(final_df['customer_city'].unique()))

Total unique: 32
['Ahmedabad', 'Aligarh', 'Allahabad', 'Bareilly', 'Bengaluru', 'Bhubaneswar', 'Calcutta', 'Chandigarh', 'Chennai', 'Coimbatore', 'Delhi', 'Gorakhpur', 'Hyderabad', 'Indore', 'Jaipur', 'Kanpur', 'Kochi', 'Kolkata', 'Lucknow', 'Ludhiana', 'Meerut', 'Moradabad', 'Mumbai', 'Nagpur', 'New Delhi', 'Patna', 'Pune', 'Saharanpur', 'Surat', 'Vadodara', 'Varanasi', 'Visakhapatnam']


In [45]:
print(sorted(final_df['delivery_type'].unique()))

['Express', 'Same Day', 'Standard']


In [46]:
if 'delivery_type' in df.columns:
    df['delivery_type'] = (
        df['delivery_type']
        .astype(str)
        .str.strip()
        .str.title()
    )

In [47]:
delivery_speed_map = {
    'Same Day': 'Fast',
    'Express': 'Fast',
    'Standard': 'Normal'
}

df['delivery_speed'] = df['delivery_type'].map(delivery_speed_map)

In [48]:
print(sorted(final_df['return_status'].unique()))

['Cancelled', 'Delivered', 'Returned']


In [49]:
if 'return_status' in df.columns:
    df['return_status'] = (
        df['return_status']
        .astype(str)
        .str.strip()
        .str.title()
    )

In [50]:
df['is_returned'] = df['return_status'].apply(
    lambda x: True if x in ['Returned', 'Cancelled'] else False
)

In [51]:
df['original_price_inr'] = df['original_price_inr'].astype(str)

df['original_price_inr'] = df['original_price_inr'].str.replace('₹', '', regex=False)
df['original_price_inr'] = df['original_price_inr'].str.replace(',', '', regex=False)

In [52]:
df['original_price_inr'] = pd.to_numeric(df['original_price_inr'], errors='coerce')

In [53]:
df = df.dropna(subset=['original_price_inr'])

In [54]:
df[df['original_price_inr'] < 0][
    ['original_price_inr', 'discounted_price_inr', 'discount_percent']
]

,original_price_inr,discounted_price_inr,discount_percent
164,-128145.60,101422.37,20.85
497,-13158.23,13158.23,0.00
577,-24345.39,20219.41,16.95
872,-43898.95,43898.95,0.00
1326,-27517.79,27517.79,0.00
...,...,...,...
75757,-33664.58,30272.80,10.08
75778,-22557.36,21237.10,5.85
76143,-99038.79,76532.21,22.73
76955,-27201.24,27201.24,0.00


In [55]:
cols = ['original_price_inr', 'discounted_price_inr']

for col in cols:
    df[col] = df[col].astype(str)
    df[col] = df[col].str.replace('₹', '', regex=False)
    df[col] = df[col].str.replace(',', '', regex=False)
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [56]:
df.loc[df['original_price_inr'] < 0, 'original_price_inr'] = \
    df.loc[df['original_price_inr'] < 0, 'original_price_inr'].abs()

In [57]:
df['discount_percent'] = (
    (df['original_price_inr'] - df['discounted_price_inr']) 
    / df['original_price_inr']
) * 100

In [58]:
# Check again
df[df['original_price_inr'] < 0]

# Check invalid discounts
df[df['discount_percent'] < 0]
df[df['discount_percent'] > 100]

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating,payment_group,delivery_speed,is_returned


In [59]:
df['price_fix_flag'] = df['original_price_inr'] < 0

In [60]:
df.head()

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating,payment_group,delivery_speed,is_returned,price_fix_flag
0,TXN_2025_00000001,2025-01-08,CUST_2025_00005600,PROD_000627,Oppo F11 Pro 128GB Black,Electronics,Smartphones,Oppo,10234.12,0.000000,...,1,2025,1,0.15,True,4.4,Digital,Normal,False,False
1,TXN_2025_00000002,01/15/2025,CUST_2022_00027099,PROD_001699,Samsung Slate 4GB RAM Silver,Electronics,Tablets,Samsung,38241.08,0.000000,...,1,2025,1,0.64,TRUE,3.4,Digital,Fast,True,False
2,TXN_2025_00000003,2025-01-26,CUST_2021_00027917,PROD_001242,Apple iPhone 16 Plus 64GB Black,Electronics,Smartphones,Apple,121974.26,32.038809,...,1,2025,1,0.18,True,3.4,Digital,Fast,True,False
3,TXN_2025_00000004,2025-01-04,CUST_2025_00004184,PROD_000979,Samsung Galaxy S22+ 128GB White,Electronics,Smartphones,Samsung,59075.70,0.000000,...,1,2025,1,0.24,False,3.3,Digital,Fast,False,False
4,TXN_2025_00000005,2025-01-03,CUST_2025_00005205,PROD_001876,Apple Watch Premium,Electronics,Smart Watch,Apple,74269.31,0.000000,...,1,2025,1,0.05,TRUE,4.1,Digital,Fast,True,False


In [61]:
df[df['discounted_price_inr'] > df['original_price_inr']]

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating,payment_group,delivery_speed,is_returned,price_fix_flag


In [62]:
df[(df['discount_percent'] < 0) | (df['discount_percent'] > 100)]

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating,payment_group,delivery_speed,is_returned,price_fix_flag


In [63]:
df['calc_total'] = (
    df['discounted_price_inr'] * df['quantity'] + df['delivery_charges']
)

df[abs(df['final_amount_inr'] - df['calc_total']) > 1]

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating,payment_group,delivery_speed,is_returned,price_fix_flag,calc_total


In [64]:
df['delivery_days'].unique()[:20]

array(['3', '2', '1', '5', '6', '4', '0', '-1', '1-2 days', '7',
       'Same Day', 'Express', '15'], dtype=object)

In [65]:
import numpy as np

def clean_delivery(x):
    x = str(x).lower().strip()
    
    if x == '-1':
        return np.nan  # invalid
    
    elif 'same' in x:
        return 0
    
    elif 'express' in x:
        return 1
    
    elif '-' in x:  # '1-2 days'
        return float(x.split('-')[0])
    
    elif 'day' in x:
        return float(x.split()[0])
    
    else:
        try:
            return float(x)
        except:
            return np.nan

In [66]:
df['delivery_days'] = df['delivery_days'].apply(clean_delivery)

In [67]:
df = df.dropna(subset=['delivery_days'])

In [68]:
df['delivery_days'] = df['delivery_days'].astype(int)

In [69]:
df['delivery_days'].unique()
df[df['delivery_days'] < 0]
df[df['delivery_days'] > 30]

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating,payment_group,delivery_speed,is_returned,price_fix_flag,calc_total


In [70]:
final_df.to_csv("../../data/processed_datasets/amazon_full_cleaned.csv", index=False)